# **Project 2: Deep generative models for personalised oncology** · ML for Healthcare, UC3M 2026/27

## 1. Load the Data

In [ ]:
import importlib.util
import pandas as pd
from packaging.version import Version
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.io
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import scanpy as sc
import scvi
import torch

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

import urllib.request
import json
import shutil
import ssl


warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False)

SEED = 0
scvi.settings.seed = SEED
rng = np.random.default_rng(SEED)

In [ ]:
# ---------------- GLOBAL CONFIG ----------------
DATA_DIR    = Path("data")
H5AD_PATH   = DATA_DIR / "wu2021_brca.h5ad"                # option A: .h5ad from CELLxGENE
GEO_DIR     = DATA_DIR / "Wu_etal_2021_BRCA_scRNASeq"      # option B: extracted GEO GSE176078 folder
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR  = Path("models");  MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
def load_wu2021():
    if not H5AD_PATH.exists() and not GEO_DIR.exists():
        print("Data not found locally, downloading it from CELLxGENE.")
        try:
            download_from_cellxgene()
        except Exception as e:
            raise FileNotFoundError(f"Automatic download failed ({e}). Download manually to {H5AD_PATH}.") from e

    if H5AD_PATH.exists():
        adata = sc.read_h5ad(H5AD_PATH)
        if not is_integer_counts(adata.X):
            assert adata.raw is not None, "X is not raw counts and there is no .raw"
            adata = adata.raw.to_adata()
        
        if "feature_name" in adata.var: # CELLxGENE uses Ensembl IDs as the index
            adata.var["ensembl_id"] = adata.var_names
            adata.var_names = adata.var["feature_name"].astype(str)
            
    elif GEO_DIR.exists():
        X = sp.mmread(GEO_DIR / "count_matrix_sparse.mtx").tocsr()
        genes = pd.read_csv(GEO_DIR / "count_matrix_genes.tsv", header=None, sep="\t")[0].astype(str).values
        barcodes = pd.read_csv(GEO_DIR / "count_matrix_barcodes.tsv", header=None, sep="\t")[0].astype(str).values
        if X.shape[0] == len(genes) and X.shape[1] == len(barcodes):
            X = X.T.tocsr()
        meta = pd.read_csv(GEO_DIR / "metadata.csv", index_col=0)
        adata = ad.AnnData(X=X, obs=meta.loc[barcodes].copy(), var=pd.DataFrame(index=genes))
    else:
        raise FileNotFoundError("Data file not found.")

    adata.var_names_make_unique()
    assert is_integer_counts(adata.X), "The matrix doesn't look like raw counts"
    return adata

def first_col(obs, candidates):
    for c in candidates:
        if c in obs.columns:
            return c
    raise KeyError(f"None of {candidates} found in obs: {list(obs.columns)}")

# processing
adata = load_wu2021()

# Standardizing names 
COLMAP = {
    "patient":        first_col(adata.obs, ["orig.ident", "donor_id", "Patient", "patient"]),
    "subtype":        first_col(adata.obs, ["subtype", "Subtype", "clinical_subtype", "disease"]),
    "celltype":       first_col(adata.obs, ["celltype_major", "author_cell_type", "cell_type"]),
    "celltype_minor": first_col(adata.obs, ["celltype_minor", "celltype_major", "author_cell_type", "cell_type"]),
}

for std, orig in COLMAP.items():
    adata.obs[std] = adata.obs[orig].astype(str).astype("category")

In [ ]:
CXG_COLLECTION_ID = "dea97145-f712-431c-a223-6b5f565f362a"
CXG_DATASET_ID    = "9fddb063-056d-4202-8b8a-4b0ee531d3ce"

# Set to None to keep the full dataset for EDA (skips subsampling)
N_CELLS = None 
SEED = 42

#  Helper Functions 
def to_dense(x):
    if sp.issparse(x):
        return x.toarray()
    if hasattr(x, "todense"):
        return np.asarray(x.todense())
    return np.asarray(x)

# We need raw counts as input to scVI and for scanpy functions
def is_integer_counts(X, n=500):
    sub = to_dense(X[:n])
    return bool(np.all(sub >= 0) and np.allclose(sub, np.round(sub)))

def ssl_context():
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except ImportError:
        return ssl.create_default_context()

In [ ]:
path = MODELS_DIR / "full_nb"
model = scvi.model.SCVI.load(path, adata=adata)

In [ ]:
# Training curves: validation ELBO
# Check convergence and potential overfitting

fig, ax = plt.subplots(figsize=(7, 4))

h = model.history

if "elbo_validation" in h:
    ax.plot(
        h["elbo_validation"].values.ravel(),
        label="scVI NB (full dataset)"
    )
else:
    print("Validation ELBO not available")

ax.set(
    xlabel="Epoch",
    ylabel="ELBO (validation)",
    title="Convergence"
)

ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
adata.obsm["X_scvi_nb"] = model.get_latent_representation(adata)

In [ ]:
# Neighbour graph
sc.pp.neighbors(
    adata,
    use_rep="X_scvi_nb",
    random_state=SEED
)

# UMAP
sc.tl.umap(
    adata,
    random_state=SEED
)

# Save UMAP coordinates
adata.obsm["X_umap_scvi_nb"] = adata.obsm["X_umap"].copy()

# Plot
sc.pl.embedding(
    adata,
    basis="X_umap_scvi_nb",
    color=["celltype", "patient", "subtype"],
    wspace=0.45,
    legend_fontsize=7,
    title=[
        "scVI NB · Cell type",
        "scVI NB · Patient",
        "scVI NB · Subtype"
    ]
)

Podemos determinar que ciertos comportamientos hacen referencia a diferentes pacientes, células, y tipos de cáncer.

Aunque esto no es suficiente para demostrar nada, vamos a centrarnos en nuestra pregunta para ir investigando cada vez más y acercarnos a nuestro objetivo.

**PREGUNTA A)** ¿Debemos considerar a cada paciente como un lote (batch) y eliminar las diferencias entre ellos durante la integración?
Esto quiere decir que cualquier diferencia entre pacientes se debe a que son diferentes pacientes y no diferencia biológica.

Como ya sabemos anteriormente cada célula cancerígena es específica para cada paciente, pero para las T-cells por ejemplo, esas diferencias entre ellas (que a priori no somos capaces de visualizar en un UMAP) se deben al paciente o realmente tienen una carga genética.

Otros tipos de células con comportamineto similar entre pacientes son (Myeloid, CAFs)

¿Cómo podemos saber si diferenciar es realemnte útil o no?


**Hipótesis 1:** ¿Podemos identificar a "X" paciente mediante sus T-cells?
Para responder esta pregunta planteamos un clasificador, pero, debemos tener cuidado a la hora de entrenarlo ya que podría estar aprendiendo diferencias no solo biológicas sino técnicas.

Si el clasificador identifica al paciente podriamos concluir que existen diferencias biológicas entre las T-cells de los pacientes, por lo tanto, estaría justificado la separación por paciente (batch effect).

## Hipótesis 1:

In [ ]:
# Clasificador de paciente en células T: antes vs después de integrar (scVI)

# Pregunta: ¿se puede identificar al paciente a partir de sus células T?
#   - Antes de integrar (PCA sobre expresión log-normalizada)
#   - Después de integrar (espacio latente de scVI)

In [ ]:
batch_table = pd.crosstab(
    adata.obs["donor_id"],
    adata.obs["batch_run"]
)

display(batch_table)

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat")   # sin batch_key a propósito
adata_hvg = X[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)

sc.pp.pca(adata_hvg, n_comps=10, random_state=0) 

X_pca = adata_hvg.obsm["X_pca"]

X = X_hvg
y = X_hvg.obs["patient"].values

X_scvi = adata.obsm["X_scvi_nb"] # anteriormente calculado con scVI adata.obsm["X_scvi_nb"] = model.get_latent_representation(a)
y_scvi = adata.obs["patient"].values

In [ ]:
# Split

Nos hemos dado cuenta de que no tenemos diferentes muestras para ver si nuestro clasificador puede detectar al paciente.

Si separo randomly mi conjunto de células podría aprender diferencias técnicas y no  biológicas por lo que tenemos que saber cómo afrontar este problema.

Podemos sacar los genes más importantes mediante el clasificador.
Utilizar las células no utilizadas y más concretamente esos genes para saber si es capaz de detectar al paciente.

Si comparas la accuracy entre pca y scvi qué te asegura que la diferencia entre uno y otro sea debido a variabilidad técnica o biológica, ver cómo afrontar este problema.